In [ ]:
!pip install -q rank-bm25 sentence-transformers faiss-gpu-cu12 networkx tqdm numpy underthesea

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value = user_secrets.get_secret("hf_token")

In [ ]:
from huggingface_hub import login
login(token=secret_value)

In [ ]:
import json
import pickle
import re
from collections import defaultdict

import faiss
import networkx as nx
import numpy as np
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from tqdm import tqdm
from underthesea import word_tokenize

In [ ]:
MODEL_NAME = "BAAI/bge-m3"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"

EMBED_BATCH = 32
RERANK_BATCH = 256

EMBEDDINGS_FILE = "/kaggle/input/notebooks/minhvb10/qa-retrieval-embed/paragraph_embeddings.pkl"
FAISS_INDEX_FILE = "/kaggle/input/notebooks/minhvb10/qa-retrieval-embed/paragraph_embeddings.faiss"

CORPUS_FILE = "/kaggle/input/datasets/minhvb10/zalo-dataset/corpus_preprocessed_v2.jsonl"
QUERIES_FILE = "/kaggle/input/datasets/minhvb10/zalo-dataset/queries_preprocessed_v2.jsonl"
TEST_FILE = "/kaggle/input/datasets/minhvb10/zalo-dataset/test.jsonl"
GRAPH_FILE = "/kaggle/input/datasets/minhvb10/zalo-dataset/legal_kg_complete.json"

SEMANTIC_TOP_K_PARAGRAPHS = 80
BM25_TOP_K_PARAGRAPHS = 30
KG_RELATION_HOPS = 1

MAX_CANDIDATE_PARAGRAPHS_BEFORE_RERANK = 200
TOP_K_FINAL = 10

SEMANTIC_WEIGHT = 0.7
BM25_WEIGHT = 0.2
KG_WEIGHT = 0.1

USE_HYBRID_RERANK_SCORE = False
RERANKER_WEIGHT = 0.70
BEFORE_FUSION_WEIGHT = 0.30

LAW_RELATION_EDGE_TYPES = {
    "REFERENCES",
    "AMENDS",
    "REPLACES",
    "DETAILS",
}
LAW_RELATION_WEIGHTS = {
    "DETAILS": 0.9,
    "REFERENCES": 0.9,
    "AMENDS": 0.9,
    "REPLACES": 0.9,
}
DEFAULT_LAW_RELATION_WEIGHT = 0.50

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def simple_tokenize(text):
    if text is None:
        return []

    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()

    if not text:
        return []

    segmented_text = word_tokenize(text, format="text")
    return segmented_text.split()


def get_bm25_document_text(item):
    for key in ["bm25text", "tokenized_text", "processed_text"]:
        value = item.get(key)
        if value and str(value).strip():
            return str(value).strip()

    return f"{item.get('title', '')} {item.get('text', '')}".strip()


def get_rerank_document_text(item):
    title = str(item.get("title", "") or "").strip()
    text = str(item.get("text", "") or "").strip()
    joined = f"{title}\n{text}".strip()

    if joined:
        return joined

    return get_bm25_document_text(item)


def get_query_texts(item):
    semantic_text = str(item.get("text", "") or "").strip()

    if not semantic_text:
        for key in ["processed_text", "bm25text", "tokenized_text"]:
            value = item.get(key)
            if value and str(value).strip():
                semantic_text = str(value).strip()
                break

    bm25_text = ""
    for key in ["bm25text", "tokenized_text", "processed_text", "text"]:
        value = item.get(key)
        if value and str(value).strip():
            bm25_text = str(value).strip()
            break

    return semantic_text, bm25_text

def extract_section_id(paragraph_id):
    match = re.match(r"^(.+?)_para_\d+$", str(paragraph_id))
    if match:
        return match.group(1)

    return str(paragraph_id)


def extract_law_id(section_id):
    match = re.match(r"^(.+?)\+\d+$", str(section_id))
    if match:
        return match.group(1)

    return str(section_id)

def load_corpus_jsonl(path):
    section_text_map = {}
    bm25_text_map = {}
    section_ids = []
    tokenized_corpus = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            item = json.loads(line)
            section_id = str(item["_id"])

            bm25_text = get_bm25_document_text(item)
            rerank_text = get_rerank_document_text(item)

            section_text_map[section_id] = rerank_text
            bm25_text_map[section_id] = bm25_text
            section_ids.append(section_id)
            tokenized_corpus.append(simple_tokenize(bm25_text))

    return section_text_map, bm25_text_map, section_ids, tokenized_corpus


def load_queries_jsonl(path):
    queries = {}

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            item = json.loads(line)
            query_id = str(item["_id"])
            semantic_text, bm25_text = get_query_texts(item)

            queries[query_id] = {
                "semantic": semantic_text,
                "bm25": bm25_text,
            }

    return queries


def load_qrels_jsonl(path):
    qrels = defaultdict(set)

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            item = json.loads(line)
            if item.get("score", 0) > 0:
                query_id = str(item["query-id"])
                section_id = str(item["corpus-id"])
                qrels[query_id].add(section_id)

    return dict(qrels)


def load_knowledge_graph(path):
    path = str(path)

    if path.endswith(".graphml"):
        return nx.read_graphml(path)

    if path.endswith(".json"):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        graph = nx.DiGraph()

        for node_data in data.get("nodes", []):
            attrs = dict(node_data)
            node_id = str(attrs.pop("id"))
            graph.add_node(node_id, **attrs)

        for edge_data in data.get("edges", []):
            attrs = dict(edge_data)
            source = str(attrs.pop("source"))
            target = str(attrs.pop("target"))
            graph.add_edge(source, target, **attrs)

        return graph

    raise ValueError(f"Unsupported graph format: {path}")


def load_paragraph_vector_store(embeddings_file, faiss_index_file):
    with open(embeddings_file, "rb") as f:
        embeddings_data = pickle.load(f)

    embeddings_dict = embeddings_data["embeddings_dict"]
    paragraph_ids = [str(pid) for pid in embeddings_data["paragraph_ids"]]
    index = faiss.read_index(str(faiss_index_file))

    if index.ntotal != len(paragraph_ids):
        raise ValueError(
            f"FAISS index has {index.ntotal} vectors but pickle has "
            f"{len(paragraph_ids)} paragraph ids"
        )

    return embeddings_dict, paragraph_ids, index

def get_edge_attr(graph, source, target):
    data = graph.get_edge_data(source, target, default={})
    if data and "edge_type" not in data and all(isinstance(v, dict) for v in data.values()):
        return next(iter(data.values()))

    return data or {}


def get_edge_type(edge_attr):
    return (
        edge_attr.get("edge_type")
        or edge_attr.get("relationship_type")
        or edge_attr.get("label")
        or edge_attr.get("type")
    )


def update_score(scores, key, score):
    if score > scores.get(key, float("-inf")):
        scores[key] = score


def paragraph_text(embeddings_dict, paragraph_id):
    info = embeddings_dict.get(paragraph_id, {})
    parts = [
        info.get("section_title", ""),
        info.get("label", ""),
        info.get("content", ""),
    ]

    return "\n".join(
        part.strip()
        for part in parts
        if isinstance(part, str) and part.strip()
    )


def build_maps(section_ids, paragraph_ids):
    section_to_paragraphs = defaultdict(list)
    law_to_sections = defaultdict(list)
    seen_law_sections = defaultdict(set)

    for paragraph_id in paragraph_ids:
        section_id = extract_section_id(paragraph_id)
        law_id = extract_law_id(section_id)

        section_to_paragraphs[section_id].append(paragraph_id)

        if section_id not in seen_law_sections[law_id]:
            law_to_sections[law_id].append(section_id)
            seen_law_sections[law_id].add(section_id)

    for section_id in section_ids:
        law_id = extract_law_id(section_id)

        if section_id not in seen_law_sections[law_id]:
            law_to_sections[law_id].append(section_id)
            seen_law_sections[law_id].add(section_id)

    return dict(section_to_paragraphs), dict(law_to_sections)


def get_parent_law(graph, section_id):
    if section_id in graph:
        node_type = graph.nodes[section_id].get("node_type")
        if node_type == "LAW":
            return section_id

        if hasattr(graph, "predecessors"):
            predecessors = graph.predecessors(section_id)
        else:
            predecessors = graph.neighbors(section_id)

        for pred in predecessors:
            edge_type = get_edge_type(get_edge_attr(graph, pred, section_id))
            if graph.nodes[pred].get("node_type") == "LAW" and edge_type == "HAS_SECTION":
                return pred

    return extract_law_id(section_id)


def iter_law_sections(graph, law_id, law_to_sections):
    if law_id in law_to_sections:
        yield from law_to_sections[law_id]
        return

    if law_id not in graph:
        return

    if hasattr(graph, "successors"):
        successors = graph.successors(law_id)
    else:
        successors = graph.neighbors(law_id)

    for child in successors:
        edge_type = get_edge_type(get_edge_attr(graph, law_id, child))
        if graph.nodes[child].get("node_type") == "SECTION" and edge_type == "HAS_SECTION":
            yield child


def iter_related_laws(graph, law_id):
    if law_id not in graph:
        return

    if hasattr(graph, "successors"):
        successors = graph.successors(law_id)
    else:
        successors = graph.neighbors(law_id)

    for target in successors:
        if target not in graph:
            continue

        edge_attr = get_edge_attr(graph, law_id, target)
        edge_type = get_edge_type(edge_attr)

        if graph.nodes[target].get("node_type") != "LAW":
            continue

        if edge_type and edge_type not in LAW_RELATION_EDGE_TYPES:
            continue

        relation_weight = LAW_RELATION_WEIGHTS.get(edge_type, DEFAULT_LAW_RELATION_WEIGHT)
        yield target, relation_weight

    if not hasattr(graph, "predecessors"):
        return

    for source in graph.predecessors(law_id):
        if source not in graph:
            continue

        edge_attr = get_edge_attr(graph, source, law_id)
        edge_type = get_edge_type(edge_attr)

        if graph.nodes[source].get("node_type") != "LAW":
            continue

        if edge_type and edge_type not in LAW_RELATION_EDGE_TYPES:
            continue

        relation_weight = LAW_RELATION_WEIGHTS.get(edge_type, DEFAULT_LAW_RELATION_WEIGHT)
        yield source, relation_weight


def section_scores_to_paragraph_scores(section_scores, section_to_paragraphs):
    paragraph_scores = {}

    for section_id, section_score in section_scores.items():
        for paragraph_id in section_to_paragraphs.get(section_id, []):
            update_score(paragraph_scores, paragraph_id, section_score)

    return paragraph_scores


def ranked_unique_sections_from_paragraphs(paragraph_ids):
    section_ids = []
    seen = set()

    for paragraph_id in paragraph_ids:
        section_id = extract_section_id(paragraph_id)
        if section_id in seen:
            continue

        section_ids.append(section_id)
        seen.add(section_id)

    return section_ids

def build_paragraph_bm25(embeddings_dict, paragraph_ids):
    tokenized_corpus = []

    for paragraph_id in paragraph_ids:
        text = paragraph_text(embeddings_dict, paragraph_id)
        tokenized_corpus.append(simple_tokenize(text))

    return BM25Okapi(tokenized_corpus)


def bm25_retrieve_paragraph_scores(bm25, paragraph_ids, query_text, top_k=BM25_TOP_K_PARAGRAPHS):
    query_tokens = simple_tokenize(query_text)
    if not query_tokens:
        return {}

    scores = bm25.get_scores(query_tokens)
    top_k = min(top_k, len(paragraph_ids))

    if top_k <= 0:
        return {}

    candidate_idx = np.argpartition(scores, -top_k)[-top_k:]
    ranked_idx = candidate_idx[np.argsort(scores[candidate_idx])[::-1]]

    result = {}
    for idx in ranked_idx:
        score = float(scores[idx])
        if score > 0:
            paragraph_id = paragraph_ids[idx]
            update_score(result, paragraph_id, score)

    return result


def embed_queries(model, texts, batch_size=EMBED_BATCH):
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        show_progress_bar=True,
    )
    embeddings = embeddings.astype(np.float32)
    faiss.normalize_L2(embeddings)
    return embeddings


def semantic_retrieve_paragraph_scores(index, paragraph_ids, q_emb_single, top_k=SEMANTIC_TOP_K_PARAGRAPHS):
    q_emb = np.array([q_emb_single], dtype=np.float32)
    faiss.normalize_L2(q_emb)

    distances, indices = index.search(q_emb, top_k)

    paragraph_scores = {}
    for idx, score in zip(indices[0], distances[0]):
        if idx == -1:
            continue

        paragraph_id = paragraph_ids[idx]
        update_score(paragraph_scores, paragraph_id, float(score))

    return paragraph_scores


def aggregate_paragraph_scores_to_sections(paragraph_scores):
    section_scores = {}

    for paragraph_id, score in paragraph_scores.items():
        section_id = extract_section_id(paragraph_id)
        update_score(section_scores, section_id, score)

    return section_scores


def normalize_score_dict(scores):
    if not scores:
        return {}

    values = np.array(list(scores.values()), dtype=np.float32)
    min_v = float(values.min())
    max_v = float(values.max())

    if abs(max_v - min_v) < 1e-12:
        return {key: 1.0 for key in scores}

    return {
        key: (float(value) - min_v) / (max_v - min_v)
        for key, value in scores.items()
    }


def expand_kg_section_scores(graph, seed_section_scores, law_to_sections, relation_hops=KG_RELATION_HOPS):
    kg_scores = {}
    law_scores = {}

    for section_id, score in seed_section_scores.items():
        law_id = get_parent_law(graph, section_id)
        update_score(law_scores, law_id, score)

    frontier = dict(law_scores)
    visited_laws = set(frontier.keys())

    for _ in range(max(0, relation_hops)):
        next_frontier = {}

        for law_id, law_score in frontier.items():
            for related_law_id, relation_weight in iter_related_laws(graph, law_id):
                related_law_score = law_score * relation_weight 

                for section_id in iter_law_sections(graph, related_law_id, law_to_sections):
                    update_score(kg_scores, section_id, related_law_score)

                if related_law_id not in visited_laws:
                    update_score(next_frontier, related_law_id, related_law_score)
                    visited_laws.add(related_law_id)

        frontier = next_frontier

        if not frontier:
            break

    return kg_scores


def hybrid_semantic_bm25_kg_paragraph_retrieve_before_after(
    index,
    paragraph_ids,
    embeddings_dict,
    bm25,
    graph,
    section_ids,
    section_to_paragraphs,
    law_to_sections,
    q_emb_single,
    semantic_query_text,
    bm25_query_text,
    reranker,
    top_k_final=TOP_K_FINAL,
):
    semantic_para_scores_raw = semantic_retrieve_paragraph_scores(
        index=index,
        paragraph_ids=paragraph_ids,
        q_emb_single=q_emb_single,
        top_k=SEMANTIC_TOP_K_PARAGRAPHS,
    )

    bm25_para_scores_raw = bm25_retrieve_paragraph_scores(
        bm25=bm25,
        paragraph_ids=paragraph_ids,
        query_text=bm25_query_text,
        top_k=BM25_TOP_K_PARAGRAPHS,
    )

    semantic_para_scores = normalize_score_dict(semantic_para_scores_raw)
    bm25_para_scores = normalize_score_dict(bm25_para_scores_raw)

    all_seed_paragraphs = set(semantic_para_scores) | set(bm25_para_scores)
    
    seed_paragraph_scores = {}
    for paragraph_id in all_seed_paragraphs:
        score = (
            SEMANTIC_WEIGHT * semantic_para_scores.get(paragraph_id, 0.0)
            + BM25_WEIGHT * bm25_para_scores.get(paragraph_id, 0.0)
        )
        seed_paragraph_scores[paragraph_id] = score
    
    seed_section_scores = aggregate_paragraph_scores_to_sections(seed_paragraph_scores)

    if not seed_section_scores:
        return [], [], [], []

    kg_section_scores_raw = expand_kg_section_scores(
        graph=graph,
        seed_section_scores=seed_section_scores,
        law_to_sections=law_to_sections,
        relation_hops=KG_RELATION_HOPS,
    )
    kg_para_scores_raw = section_scores_to_paragraph_scores(
        kg_section_scores_raw,
        section_to_paragraphs,
    )
    kg_para_scores = kg_para_scores_raw

    all_candidate_paragraphs = (
        set(semantic_para_scores)
        | set(bm25_para_scores)
        | set(kg_para_scores)
    )

    final_para_scores = {}
    for paragraph_id in all_candidate_paragraphs:
        score = (
            SEMANTIC_WEIGHT * semantic_para_scores.get(paragraph_id, 0.0)
            + BM25_WEIGHT * bm25_para_scores.get(paragraph_id, 0.0)
            + KG_WEIGHT * kg_para_scores.get(paragraph_id, 0.0)
        )
        final_para_scores[paragraph_id] = score

    before_rerank_paragraph_ids = sorted(
        final_para_scores,
        key=lambda paragraph_id: final_para_scores[paragraph_id],
        reverse=True,
    )

    before_rerank_section_ids = ranked_unique_sections_from_paragraphs(before_rerank_paragraph_ids)

    candidate_paragraph_ids = before_rerank_paragraph_ids[:MAX_CANDIDATE_PARAGRAPHS_BEFORE_RERANK]

    cross_inputs = []
    valid_paragraph_ids = []

    for paragraph_id in candidate_paragraph_ids:
        text = paragraph_text(embeddings_dict, paragraph_id)

        if not text:
            continue

        cross_inputs.append([semantic_query_text, text])
        valid_paragraph_ids.append(paragraph_id)

    if not cross_inputs:
        return (
            before_rerank_section_ids[:top_k_final],
            before_rerank_paragraph_ids[:MAX_CANDIDATE_PARAGRAPHS_BEFORE_RERANK],
            before_rerank_section_ids[:top_k_final],
            before_rerank_paragraph_ids[:MAX_CANDIDATE_PARAGRAPHS_BEFORE_RERANK],
        )

    rerank_scores = reranker.predict(
        cross_inputs,
        batch_size=RERANK_BATCH,
        show_progress_bar=False,
    )

    rerank_para_scores = {
        paragraph_id: float(score)
        for paragraph_id, score in zip(valid_paragraph_ids, rerank_scores)
    }
    rerank_para_scores_norm = normalize_score_dict(rerank_para_scores)

    if USE_HYBRID_RERANK_SCORE:
        after_para_scores = {}
        for paragraph_id in valid_paragraph_ids:
            after_para_scores[paragraph_id] = (
                RERANKER_WEIGHT * rerank_para_scores_norm.get(paragraph_id, 0.0)
                + BEFORE_FUSION_WEIGHT * final_para_scores.get(paragraph_id, 0.0)
            )
    else:
        after_para_scores = rerank_para_scores_norm

    after_rerank_paragraph_ids = sorted(
        valid_paragraph_ids,
        key=lambda paragraph_id: after_para_scores[paragraph_id],
        reverse=True,
    )

    after_rerank_section_ids = ranked_unique_sections_from_paragraphs(after_rerank_paragraph_ids)

    return (
        before_rerank_section_ids[:top_k_final],
        before_rerank_paragraph_ids[:MAX_CANDIDATE_PARAGRAPHS_BEFORE_RERANK],
        after_rerank_section_ids[:top_k_final],
        after_rerank_paragraph_ids[:MAX_CANDIDATE_PARAGRAPHS_BEFORE_RERANK],
    )

def reciprocal_rank_at_k(ranked_ids, gt_ids, k):
    for rank, item_id in enumerate(ranked_ids[:k], start=1):
        if item_id in gt_ids:
            return 1.0 / rank
    return 0.0


def init_metrics(k_list):
    return {
        "hits": {k: 0 for k in k_list},
        "recalls": {k: 0.0 for k in k_list},
        "precisions": {k: 0.0 for k in k_list},
        "f1_scores": {k: 0.0 for k in k_list},
        "mrrs": {k: 0.0 for k in k_list},
    }


def update_metrics(metrics, ranked_section_ids, gt_section_ids, k_list):
    for k in k_list:
        preds = set(ranked_section_ids[:k])
        correct = preds.intersection(gt_section_ids)
        num_correct = len(correct)

        if num_correct > 0:
            metrics["hits"][k] += 1

        recall = num_correct / len(gt_section_ids)
        precision = num_correct / k
        mrr = reciprocal_rank_at_k(ranked_section_ids, gt_section_ids, k)

        metrics["recalls"][k] += recall
        metrics["precisions"][k] += precision
        metrics["mrrs"][k] += mrr

        if recall + precision > 0:
            metrics["f1_scores"][k] += 2 * recall * precision / (recall + precision)


def print_metrics_table(title, metrics, num_queries, k_list):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    if num_queries == 0:
        print("No queries with ground truth to evaluate.")
        return

    print(
        f"{'K':>3} | "
        f"{'Recall':>8} | "
        f"{'Precision':>9} | "
        f"{'F1':>8} | "
        f"{'Hit':>8} | "
        f"{'MRR':>8}"
    )
    print("-" * 100)

    for k in k_list:
        print(
            f"{k:>3d} | "
            f"{metrics['recalls'][k] / num_queries:8.4f} | "
            f"{metrics['precisions'][k] / num_queries:9.4f} | "
            f"{metrics['f1_scores'][k] / num_queries:8.4f} | "
            f"{metrics['hits'][k] / num_queries:8.4f} | "
            f"{metrics['mrrs'][k] / num_queries:8.4f}"
        )

def run_evaluation(
    index,
    paragraph_ids,
    embeddings_dict,
    bm25,
    graph,
    section_ids,
    section_to_paragraphs,
    law_to_sections,
    eval_queries,
    eval_q_embs,
    qrels,
    reranker,
):
    k_list = [1, 5, 10]

    before_metrics = init_metrics(k_list)
    after_metrics = init_metrics(k_list)

    num_queries = 0

    for i in tqdm(range(len(eval_queries)), desc="Evaluating Semantic + paragraph-BM25 + KG paragraph rerank"):
        query_id, semantic_query_text, bm25_query_text = eval_queries[i]
        q_emb = eval_q_embs[i]
        gt_ids = qrels[query_id]

        (
            before_section_ids,
            before_paragraph_ids,
            after_section_ids,
            after_paragraph_ids,
        ) = hybrid_semantic_bm25_kg_paragraph_retrieve_before_after(
            index=index,
            paragraph_ids=paragraph_ids,
            embeddings_dict=embeddings_dict,
            bm25=bm25,
            graph=graph,
            section_ids=section_ids,
            section_to_paragraphs=section_to_paragraphs,
            law_to_sections=law_to_sections,
            q_emb_single=q_emb,
            semantic_query_text=semantic_query_text,
            bm25_query_text=bm25_query_text,
            reranker=reranker,
            top_k_final=TOP_K_FINAL,
        )

        update_metrics(before_metrics, before_section_ids, gt_ids, k_list)
        update_metrics(after_metrics, after_section_ids, gt_ids, k_list)
        num_queries += 1

        if i < SHOW_EXAMPLES:
            print_example(
                query_id=query_id,
                query_text=semantic_query_text,
                gt_ids=gt_ids,
                before_section_ids=before_section_ids,
                before_paragraph_ids=before_paragraph_ids,
                after_section_ids=after_section_ids,
                after_paragraph_ids=after_paragraph_ids,
                embeddings_dict=embeddings_dict,
            )

    print_metrics_table(
        "SEMANTIC + PARAGRAPH-BM25 + KG BEFORE PARAGRAPH-LEVEL CROSS-ENCODER RERANKER",
        before_metrics,
        num_queries,
        k_list,
    )

    print_metrics_table(
        "SEMANTIC + PARAGRAPH-BM25 + KG AFTER PARAGRAPH-LEVEL CROSS-ENCODER RERANKER",
        after_metrics,
        num_queries,
        k_list,
    )

def main():
    print("1. Loading corpus, queries, qrels...")
    section_text_map, bm25_text_map, section_ids, tokenized_corpus = load_corpus_jsonl(CORPUS_FILE)
    queries = load_queries_jsonl(QUERIES_FILE)
    qrels = load_qrels_jsonl(TEST_FILE)

    eval_queries = [
        (query_id, queries[query_id]["semantic"], queries[query_id]["bm25"])
        for query_id in qrels
        if query_id in queries
    ]

    print(f"Loaded {len(section_ids)} sections from corpus")
    print(f"Loaded {len(queries)} queries; evaluating {len(eval_queries)} queries with qrels")

    print("\n2. Loading paragraph vector store and FAISS index...")
    embeddings_dict, paragraph_ids, index = load_paragraph_vector_store(
        EMBEDDINGS_FILE,
        FAISS_INDEX_FILE,
    )
    print(f"Loaded {len(paragraph_ids)} paragraphs")
    print(f"Loaded FAISS index with {index.ntotal} vectors")

    print("\n3. Building paragraph-level BM25 index...")
    bm25 = build_paragraph_bm25(embeddings_dict, paragraph_ids)
    print(f"Paragraph-level BM25 index is ready with {len(paragraph_ids)} paragraphs")

    print("\n4. Loading Knowledge Graph...")
    graph = load_knowledge_graph(GRAPH_FILE)
    section_to_paragraphs, law_to_sections = build_maps(section_ids, paragraph_ids)
    print(f"Loaded KG: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
    print(f"Built maps: {len(section_to_paragraphs)} sections with paragraphs, {len(law_to_sections)} laws")

    print("\n5. Loading embedding model for queries...")
    embedding_model = SentenceTransformer(MODEL_NAME, device=DEVICE)
    if DEVICE == "cuda":
        embedding_model.half()

    eval_query_texts = [semantic_text for _, semantic_text, _ in eval_queries]
    print("Embedding queries...")
    eval_q_embs = embed_queries(embedding_model, eval_query_texts)

    print("\n6. Loading CrossEncoder reranker...")
    reranker = CrossEncoder(RERANK_MODEL, max_length=1024, device=DEVICE)
    if DEVICE == "cuda":
        reranker.model.half()

    print("\n7. Starting evaluation...")
    print(
        f"Settings: SEMANTIC_TOP_K_PARAGRAPHS={SEMANTIC_TOP_K_PARAGRAPHS}, "
        f"BM25_TOP_K_PARAGRAPHS={BM25_TOP_K_PARAGRAPHS}, "
        f"KG_RELATION_HOPS={KG_RELATION_HOPS}, "
        f"MAX_CANDIDATE_PARAGRAPHS_BEFORE_RERANK={MAX_CANDIDATE_PARAGRAPHS_BEFORE_RERANK}"
    )
    print(
        f"Weights: semantic={SEMANTIC_WEIGHT}, bm25={BM25_WEIGHT}, kg={KG_WEIGHT}"
    )
    print(
        f"Rerank mode: paragraph-level, "
        f"USE_HYBRID_RERANK_SCORE={USE_HYBRID_RERANK_SCORE}, "
        f"reranker_weight={RERANKER_WEIGHT}, before_fusion_weight={BEFORE_FUSION_WEIGHT}"
    )

    run_evaluation(
        index=index,
        paragraph_ids=paragraph_ids,
        embeddings_dict=embeddings_dict,
        bm25=bm25,
        graph=graph,
        section_ids=section_ids,
        section_to_paragraphs=section_to_paragraphs,
        law_to_sections=law_to_sections,
        eval_queries=eval_queries,
        eval_q_embs=eval_q_embs,
        qrels=qrels,
        reranker=reranker,
    )


if __name__ == "__main__":
    main()